# 05c — Weather Dimension Table

Standalone fetch of hourly historical weather from the Open-Meteo archive API
(ERA5 reanalysis, no API key required) for a single Brisbane CBD point,
written as its own dimension table at
`s3://<AWS_S3_BUCKET>/weather/era5/hourly.parquet`.

This notebook is **independent** of notebooks 05/06/07 and the
`ml_features` snapshot — it does not read from or write to any of those, and
joining weather onto the feature snapshot is left for a future notebook.

Key steps:
1. Load `.env` / configure S3 access (same pattern as notebook 07 Cell 1).
2. Fetch hourly `temperature_2m`, `precipitation`, `wind_speed_10m`,
   `wind_gusts_10m` for `latitude=-27.4705, longitude=153.0260` (Brisbane
   CBD — single corridor point for v0) from `start_date` to today.
3. Derive `weather_date`, `weather_hour`, and `is_raining` (`precipitation
   > 0.2`).
4. Write a single parquet to S3, skipping the fetch if it already exists
   (unless `REFRESH=True`).

ERA5 reanalysis typically lags ~5 days behind real time, so the most recent
1-2 dates may have missing hours — this is expected and just reported, not
treated as an error.

In [1]:
# Cell 1 — Environment setup: load .env, configure S3 access, display options
import os
import subprocess
import sys
from pathlib import Path

try:
    from dotenv import load_dotenv, find_dotenv
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'python-dotenv', '-q'], check=True)
    from dotenv import load_dotenv, find_dotenv

try:
    import s3fs
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 's3fs', '-q'], check=True)
    import s3fs

import requests
import numpy as np
import pandas as pd

_dotenv_path = find_dotenv(usecwd=True)
if _dotenv_path:
    load_dotenv(_dotenv_path, override=False)
    print(f'Loaded .env from: {_dotenv_path}')
else:
    print('No .env file found — using defaults.')

S3_BUCKET = os.environ.get('AWS_S3_BUCKET', '')
AWS_REGION = os.environ.get('AWS_REGION', 'ap-southeast-2')
REPO_DIR   = os.environ.get('TRANSIT_AI_REPO_DIR', str(Path.cwd().parent))

if not S3_BUCKET:
    raise EnvironmentError('AWS_S3_BUCKET is not set. Check your .env file.')

fs = s3fs.S3FileSystem()

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print(f'S3_BUCKET: {S3_BUCKET}')
print(f'AWS_REGION: {AWS_REGION}')
print(f'REPO_DIR: {REPO_DIR}')
print(f's3fs: {s3fs.__version__}')

Loaded .env from: /Users/proteeksanyal/Desktop/Learning/Transit-AI/.env
S3_BUCKET: seq-transit-ai-data-ps
AWS_REGION: ap-southeast-2
REPO_DIR: /Users/proteeksanyal/Desktop/Learning/Transit-AI
s3fs: 2025.10.0


## Fetch configuration

Single corridor point (Brisbane CBD) for v0 — no per-stop weather yet.
`END_DATE` defaults to today; `REFRESH` controls whether an existing parquet
is overwritten.

In [2]:
# Cell 2 — Fetch configuration
import datetime as dt

LATITUDE = -27.4705
LONGITUDE = 153.0260  # Brisbane CBD — single corridor point for v0
START_DATE = '2026-06-28'
END_DATE = dt.date.today().isoformat()
TIMEZONE = 'Australia/Brisbane'
HOURLY_VARS = ['temperature_2m', 'precipitation', 'wind_speed_10m', 'wind_gusts_10m']

REFRESH = False  # set True to force a re-fetch even if the parquet already exists

WEATHER_S3_PATH = f's3://{S3_BUCKET}/weather/era5/hourly.parquet'

print(f'Point: ({LATITUDE}, {LONGITUDE})')
print(f'Date range requested: {START_DATE} .. {END_DATE}')
print(f'Target: {WEATHER_S3_PATH}')
print(f'REFRESH: {REFRESH}')

Point: (-27.4705, 153.026)
Date range requested: 2026-06-28 .. 2026-07-18
Target: s3://seq-transit-ai-data-ps/weather/era5/hourly.parquet
REFRESH: False


## Fetch from Open-Meteo (skip if already on S3)

`GET https://archive-api.open-meteo.com/v1/archive` — no API key required.
IAM/credential errors on the `fs.exists` check surface here immediately;
stop and report if one occurs rather than working around it.

In [3]:
# Cell 3 — Fetch hourly weather, or skip and load the existing parquet
if fs.exists(WEATHER_S3_PATH) and not REFRESH:
    print(f'{WEATHER_S3_PATH} already exists — skipping fetch (set REFRESH=True to force).')
    with fs.open(WEATHER_S3_PATH, 'rb') as f:
        weather_df = pd.read_parquet(f)
    FETCHED = False
else:
    OPEN_METEO_URL = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude': LATITUDE,
        'longitude': LONGITUDE,
        'start_date': START_DATE,
        'end_date': END_DATE,
        'timezone': TIMEZONE,
        'hourly': ','.join(HOURLY_VARS),
    }
    resp = requests.get(OPEN_METEO_URL, params=params, timeout=60)
    resp.raise_for_status()
    hourly = resp.json()['hourly']
    FETCHED = True
    print(f'Fetched {len(hourly["time"]):,} hourly records from Open-Meteo archive API.')

s3://seq-transit-ai-data-ps/weather/era5/hourly.parquet already exists — skipping fetch (set REFRESH=True to force).


## Build dimension columns

`weather_date` / `weather_hour` split out of the returned local timestamp;
`is_raining` uses a nullable boolean so hours with missing `precipitation`
(ERA5 lag) stay distinguishable from confirmed dry hours rather than
silently defaulting to `False`.

In [4]:
# Cell 4 — Build weather_date / weather_hour / is_raining
if FETCHED:
    ts = pd.to_datetime(hourly['time'])
    weather_df = pd.DataFrame({
        'weather_date': ts.strftime('%Y-%m-%d'),
        'weather_hour': ts.hour.astype('int32'),
        'temperature_2m': hourly['temperature_2m'],
        'precipitation': hourly['precipitation'],
        'wind_speed_10m': hourly['wind_speed_10m'],
        'wind_gusts_10m': hourly['wind_gusts_10m'],
    })
    weather_df['is_raining'] = weather_df['precipitation'].apply(
        lambda p: (p > 0.2) if pd.notna(p) else pd.NA
    ).astype('boolean')

    print(weather_df.dtypes)
    weather_df.head()

## Report actual date coverage

ERA5 lags real time by ~5 days, so the tail of the requested range may be
missing — expected, and reported here rather than treated as a failure.

In [5]:
# Cell 5 — Print actual date coverage vs. requested range
if FETCHED:
    unique_dates = sorted(weather_df['weather_date'].unique())
    expected_dates = pd.date_range(START_DATE, END_DATE, freq='D').strftime('%Y-%m-%d').tolist()
    missing_dates = [d for d in expected_dates if d not in unique_dates]

    print(f'Requested range: {START_DATE} .. {END_DATE} ({len(expected_dates)} days)')
    print(f'Dates returned:  {unique_dates[0]} .. {unique_dates[-1]} ({len(unique_dates)} distinct dates)')
    print(f'Rows: {len(weather_df):,}')

    if missing_dates:
        print(f'\nMissing dates ({len(missing_dates)}): {missing_dates}')
        print('Expected — ERA5 reanalysis typically lags ~5 days behind real time.')
    else:
        print('\nNo missing dates in the requested range.')

    null_counts = weather_df[HOURLY_VARS].isna().sum()
    print('\nNull counts per hourly variable (within returned rows):')
    print(null_counts.to_string())
else:
    print(f'Loaded existing parquet — {len(weather_df):,} rows, '
          f'dates {weather_df["weather_date"].min()} .. {weather_df["weather_date"].max()}')

Loaded existing parquet — 504 rows, dates 2026-06-28 .. 2026-07-18


## Write to S3

Single parquet file, only written when a fresh fetch happened this run.

In [6]:
# Cell 6 — Write the weather dimension parquet to S3
if FETCHED:
    with fs.open(WEATHER_S3_PATH, 'wb') as f:
        weather_df.to_parquet(f, index=False)
    print(f'Wrote {len(weather_df):,} rows to {WEATHER_S3_PATH}')
else:
    print('Fetch was skipped (parquet already existed) — nothing written this run.')

Fetch was skipped (parquet already existed) — nothing written this run.
